# 06. 写好提示词

AGDR VLA 入门教程 · 第 3 章 · 后半周

- **学习目标**：能写出包含背景、目标、约束、验收标准的提示词；掌握先计划后执行、完整报错、给出示例等常用技巧；能识别并改写常见的低效提示词
- **前置知识**：前半周的 01–05 节（至少学完 01、02、03）
- **预计学习时长**：1 小时

## 为什么提示词这么重要

回想前半周学到的原理：模型是无状态的，它知道的一切，**只有上下文里的内容**。系统提示词、`AGENTS.md`、它自己用工具读到的文件，再加上你的这句话，就是它的全部信息。

它不知道：

- 你在做什么项目、这一步在整个项目里的位置
- 你心里的「做好了」是什么样子
- 哪些文件不能动、哪些方案你已经试过了
- 你的电脑是什么系统、装了什么版本

可以把 Agent 想象成一位**能力很强、但今天第一天来实验室的新同学**。你怎样给新同学交代任务，就怎样给 Agent 写提示词。

> 后半周开始，本章转向「怎样用好 Agent」。这一节的大部分练习，都要在你自己安装的 Agent 里完成。

## 好提示词的四个要素

| 要素 | 回答的问题 | 例子 |
| --- | --- | --- |
| **背景** | 这是什么、在什么情况下 | 「这是 SO-100 机械臂抓取策略的训练日志，用 LeRobot 训练的」 |
| **目标** | 要做成什么 | 「判断这次训练的结果能不能拿去做真机测试」 |
| **约束** | 不能做什么、必须怎么做 | 「只读文件，不要修改；不要运行训练命令」 |
| **验收标准** | 怎样算做完了、结果长什么样 | 「给出能 / 不能的结论，列出最多 3 条依据，每条引用日志里的具体行」 |

不是每次都要把四项写满。一个简单的问题，一句话就够了。但任务越复杂、越容易做偏，就越要写全。其中**验收标准**最常被遗漏，也最有用：它同时告诉了 Agent「做到哪里停」和「用什么格式交付」。

## 对比：同一个任务，两种问法

下面用 02 节的 mini agent 做一个对比。两次问的是同一件事，只是提示词不同。

In [ ]:
import mini_agent as ma

ma.setup_workspace()
print("模式:", "在线" if ma.ONLINE else "离线（读取录制的模型响应）")

In [ ]:
vague = "看看日志有没有问题。"

answer, _ = ma.run_agent(vague, offline_script=["prompt_vague_read.json", "prompt_vague_answer.json"])
print("\n最终回答:", answer)

In [ ]:
clear = (
    "背景：工作区里的 train.log 是 SO-100 机械臂抓取策略的训练日志，配置在 config.json。\n"
    "目标：判断这次训练的结果能不能直接拿去做真机测试。\n"
    "约束：只读文件，不要运行任何命令。\n"
    "验收标准：先给出「能 / 不能」的结论，再列出最多 3 条依据，每条都引用日志或配置里的具体内容；"
    "如果仅凭这些文件无法判断，要明确说出来。"
)

answer, _ = ma.run_agent(clear, offline_script=["prompt_clear_read.json", "prompt_clear_answer.json"])
print("\n最终回答:\n")
print(answer)

对比两次的结果：

| | 模糊的提示词 | 清晰的提示词 |
| --- | --- | --- |
| 读了哪些文件 | 只读了日志 | 日志和配置都读了，因为背景里提到了 config.json |
| 结论 | 「看起来基本正常」 | 明确的「不建议直接用」，并给出替代方案 |
| 依据 | 没有 | 3 条，每条都能回到原文核对 |
| 局限 | 没有提 | 主动说明「训练日志无法判断真机效果」 |

第一次的回答不能说错，只是没什么用。问题不在模型，而在于它**不知道你真正想知道什么**。

> 在线模式下，模型的回答每次都会有些不同，但两种提示词之间的差距通常很明显。也可以试试只补上四个要素中的一个，看看哪一个影响最大。

## 常用技巧

### 1. 先计划，后执行

对于复杂的任务，先让 Agent **只出方案、不动手**：

> 我想给数据加载部分加上缓存，减少每个 epoch 读取硬盘的时间。**先不要改代码**，读一下 `dataset.py` 和 `train.py`，告诉我你打算怎么改、要改哪些文件、可能有什么风险。

看完方案，你可以纠正它的误解、补充它不知道的信息，确认后再让它执行。在方案阶段纠正一个误解，比在它改完十个文件后再推倒重来便宜得多。

不少 Agent 工具提供专门的「计划模式」，在这个模式下 Agent 只能读、不能改。具体用法见你所用工具的文档。

### 2. 一次只做一件事

「修复这个 bug，顺便重构一下，再加几个测试，然后更新文档」，这样的任务很容易做乱，出了问题也不知道是哪一步造成的。拆开来一件一件做，每做完一件就检查、提交（第 08 节）。

### 3. 报错要贴完整

只贴最后一行 `RuntimeError: CUDA out of memory`，Agent 只能泛泛地建议「减小 batch size」。好的报错求助包括：

| 内容 | 为什么 |
| --- | --- |
| 你运行的**完整命令** | 参数可能就是问题所在 |
| **完整的**报错输出，尤其是 Traceback 的开头 | 真正的原因往往在前面，最后一行只是结果 |
| 环境：系统、Python 版本、关键库的版本、GPU 型号 | 很多问题和版本有关 |
| 你**已经试过**什么，结果如何 | 避免它把你试过的方法再推荐一遍 |

如果 Agent 能自己运行命令，更好的做法是让它**自己复现**：「运行 `python train.py --config configs/so100.yaml`，看看报什么错」。它直接看到的输出，比你转述的更完整。

### 4. 给出例子

想要特定的格式时，给一个例子比描述十句话都管用：

> 把 `results/` 下每次实验的结果整理成表格，格式像这样：
>
> | 实验 | 学习率 | 成功率 |
> | --- | --- | --- |
> | exp01 | 3e-4 | 62% |

### 5. 指明位置

「那个处理图像的函数」不如「`src/vision/preprocess.py` 里的 `resize_and_crop`」。多数 Agent 工具支持用 `@` 加文件路径直接引用文件，具体写法见工具文档。即使 Agent 能自己搜索，指明位置也能省下好几轮工具调用，回想 03 节，每一轮都在消耗上下文。

### 6. 允许它说「不知道」

明确告诉它：「不确定的地方直接说不确定，不要猜」「如果信息不够，先问我」。模型默认倾向于给出一个完整的回答，哪怕它其实并不确定（回想 01 节的幻觉）。

### 7. 要求它自己验证

「改完后运行 `pytest tests/test_dataset.py`，把结果贴出来」。让验证成为验收标准的一部分，你拿到的就不是「我改好了」，而是「我改好了，这是测试通过的证据」。

### 8. 发现跑偏，立刻打断

看到 Agent 开始做你没让它做的事，不要等它做完。多数工具都可以随时中断当前操作，然后用一句话纠正方向。跑偏的时间越长，要撤销的东西就越多。

## 常见的低效提示词

| 低效的写法 | 问题 | 改进方向 |
| --- | --- | --- |
| 「帮我修一下」 | 修什么？什么现象？ | 说清现象、期望的行为、怎样复现 |
| 「优化一下这段代码」 | 优化速度？可读性？内存？ | 说清目标和衡量方式，例如「每个 epoch 的数据加载时间降到 1 分钟以内」 |
| 一条消息里塞十个需求 | 容易遗漏，出错难以定位 | 拆开，一次一件 |
| 只贴最后一行报错 | 丢失了关键信息 | 贴完整的命令和输出，或者让它自己复现 |
| 「还是不行」「不对」 | 它不知道哪里不对 | 说清这次的现象和上次有什么不同 |
| 全部大写、重复强调「一定要！！！」 | 加重语气不会增加信息 | 解释**为什么**有这条要求，模型更容易理解和遵守 |
| 每次都重新交代一遍项目背景 | 浪费时间，还容易漏 | 写进 `AGENTS.md`（第 07 节） |
| 把 API key、密码贴进对话 | 泄露风险 | 永远不要这样做（第 09 节） |

## 几个常用模板

不必死记，理解每一项为什么存在即可。

**排错求助**

```text
我在 <做什么> 时遇到了问题。
运行的命令：<完整命令>
报错输出：<完整输出>
环境：<系统 / Python 版本 / 关键库版本>
已经试过：<做过什么，结果如何>
请先分析可能的原因，按可能性从高到低列出，先不要改代码。
```

**写新功能**

```text
背景：<项目是什么，相关代码在哪>
目标：<要实现什么>
约束：<不能改哪些文件 / 必须兼容什么 / 代码风格>
验收标准：<怎样算完成，例如哪些测试要通过>
先给我方案，确认后再实现。
```

**读懂代码**

```text
我刚接手这个项目，想弄清 <某个流程>。
请从 <入口文件> 开始，告诉我数据是怎样一步步流动的，
每一步指出对应的文件和函数名。不要修改任何文件。
```

## 提示词也需要迭代

很少有提示词一次就完美。Agent 的回答不理想时，先想一想：**它误解了什么？缺了哪条信息？** 然后补上那条信息，而不是把原来的话再说一遍。

如果你发现自己总是在补充同样的信息，比如「我们用 Python 3.10」「测试用 pytest」「不要改 `third_party/` 目录」，就说明这些内容该写进项目说明文件了。这是下一节的内容。

## 动手试试

不计分。第 1–3 题只需要动笔，第 4 题要用到你自己的 Agent。

1. 下面三条提示词各缺了四要素中的哪些？把它们改写成更好的版本。
   - a. 「帮我写个脚本处理数据。」
   - b. 「我的 ROS2 节点收不到消息，怎么办？」
   - c. 「把这个仓库的 README 写得好一点。」
2. 你运行 `python train.py` 时，终端输出了 40 行 Traceback，最后一行是 `ModuleNotFoundError: No module named 'lerobot'`。按「排错求助」模板写一条提示词（Traceback 可以用「（完整输出）」代替）。
3. 下面这条提示词有什么问题？「**一定一定不要**修改 tests 文件夹！！！修一下 bug，再把代码重构得更好，顺便加点注释，然后跑一下测试。」
4. 在你自己的 Agent 里，挑本仓库的一个文件（例如第 2 章的某个 notebook），先用一句模糊的话请它讲解，再用「读懂代码」模板请它讲解，比较两次回答。**只让它读，不要让它修改任何文件。**

<details>
<summary>参考答案</summary>

**第 1 题**（改写方式不唯一，下面只是示例）

- a. 背景、目标、约束、验收标准全都缺。改写：「`data/raw/` 下有 200 个 `.json` 文件，每个是一次抓取演示，格式见 `data/raw/demo_001.json`。写一个 Python 脚本，把它们合并成一个 CSV，每行一帧，包含时间戳和 6 个关节角度。不要修改原始文件。完成后用 `demo_001.json` 测试，告诉我输出了多少行。」
- b. 缺背景和验收标准，也没有任何可以排查的信息。改写：「我有两个 ROS2 节点，`camera_node` 发布 `/image_raw`，`policy_node` 订阅它，但 `policy_node` 的回调从来没被调用。`ros2 topic list` 里能看到 `/image_raw`。相关代码在 `src/policy_node.py`。请先帮我列出可能的原因和对应的检查命令，我自己去运行。」
- c. 缺目标和验收标准：「好一点」是指什么？改写：「这个仓库的 README 只有一行。请补充：项目简介、安装步骤、怎样运行示例、目录结构说明。安装步骤要以 `requirements.txt` 为准，不要编造不存在的命令。」

**第 2 题**（示例）

```text
我在运行训练脚本时遇到了问题。
运行的命令：python train.py
报错输出：（完整输出）
环境：Ubuntu 22.04，Python 3.10，用 conda 建了一个叫 vla 的环境
已经试过：pip list 里看不到 lerobot
请先分析可能的原因，按可能性从高到低列出，先不要改代码，也不要直接帮我安装。
```

这类报错常见的原因包括：没有激活正确的环境、包装在了另一个 Python 里、包根本没装。第 4 章会讲 Python 环境。

**第 3 题**：问题有三个。第一，一条消息里塞了四件事（修 bug、重构、加注释、跑测试），而且「修 bug」没说是什么 bug。第二，「重构得更好」「加点注释」没有标准。第三，用全大写的「一定一定」和感叹号来强调约束，却没有说明原因。改进：一次只做修 bug 这一件事，说清现象和复现方法；解释为什么不能改测试（例如「测试是助教写的评分标准」）；把「运行测试并贴出结果」作为验收标准。如果「不改 tests」是长期规则，就写进 `AGENTS.md`，或者用 Hook 强制执行（第 05 节）。

**第 4 题**：开放题。通常第二次的回答会更有条理，并且会指出具体的文件和单元格。

</details>

## 小结

| 你应该能 | 相关内容 |
| --- | --- |
| 解释为什么 Agent 需要你提供背景信息 | 为什么提示词这么重要 |
| 说出好提示词的四个要素，并指出一条提示词缺了哪些 | 四个要素 |
| 在复杂任务中使用「先计划后执行」 | 技巧 1 |
| 写出一条信息完整的排错求助 | 技巧 3、模板 |
| 识别并改写常见的低效提示词 | 常见的低效提示词 |
| 判断哪些反复出现的信息应该写进项目说明文件 | 提示词也需要迭代 |

下一节：[07-项目说明文件](./07-项目说明文件.ipynb)